# DATA ENGINEERING LIFE CYCLE WITH SQL

### Step 1: Generation

In [1]:
# Imports
import pandas as pd
import sqlite3

# Read CSV into DataFrame
df = pd.read_csv('customer_data.csv', index_col=0)
df.head()

,customerName,contactLastName,contactFirstName,phone,addressLine1,addressLine2,city,state,postalCode,country,salesRepEmployeeNumber,creditLimit
customerNumber,,,,,,,,,,,,
103,Atelier graphique,Schmitt,Carine,40.32.2555,"54, rue Royale",NaN,Nantes,NaN,44000,France,1370.0,21000.0
112,Signal Gift Stores,King,Jean,7025551838,8489 Strong St.,NaN,Las Vegas,NV,83030,USA,1166.0,71800.0
114,"Australian Collectors, Co.",Ferguson,Peter,03 9520 4555,636 St Kilda Road,Level 3,Melbourne,Victoria,3004,Australia,1611.0,117300.0
119,La Rochelle Gifts,Labrune,Janine,40.67.8555,"67, rue des Cinquante Otages",NaN,Nantes,NaN,44000,France,1370.0,118200.0
121,Baane Mini Imports,Bergulfsen,Jonas,07-98 9555,Erling Skakkes gate 78,NaN,Stavern,NaN,4110,Norway,1504.0,81700.0


### Step 2: Store Data

In [2]:
# 2. Create a connection to a new SQLite database thus creating it
conn = sqlite3.connect('customer_database.db')
c = conn.cursor()

In [7]:
# Create a table to store the customer data
c.execute('''CREATE TABLE IF NOT EXISTS customers
             (customerNumber INTEGER PRIMARY KEY,
             customerName TEXT,
             contactLastName TEXT,
             contactFirstName TEXT,
             phone TEXT,
             addressLine1 TEXT,
             addressLine2 TEXT,
             city TEXT,
             state TEXT,
             postalCode INTEGER,
             country TEXT,
             saleRepEmployeeNumber INTEGER
             creditLimit INTEGER)''')

### Step 3: Ingest Data

In [8]:
#### Load/store the data from the DataFrame into the database table
df.to_sql('customers', conn, if_exists='replace', index=False)

# Sanity Check
pd.read_sql('''SELECT * FROM customers''', conn).head()

,customerNumber,customerName,contactLastName,contactFirstName,phone,addressLine1,addressLine2,city,state,postalCode,country,salesRepEmployeeNumber,creditLimit
0,103,Atelier graphique,Schmitt,Carine,40.32.2555,"54, rue Royale",None,Nantes,None,44000,France,1370.0,21000.0
1,112,Signal Gift Stores,King,Jean,7025551838,8489 Strong St.,None,Las Vegas,NV,83030,USA,1166.0,71800.0
2,114,"Australian Collectors, Co.",Ferguson,Peter,03 9520 4555,636 St Kilda Road,Level 3,Melbourne,Victoria,3004,Australia,1611.0,117300.0
3,119,La Rochelle Gifts,Labrune,Janine,40.67.8555,"67, rue des Cinquante Otages",None,Nantes,None,44000,France,1370.0,118200.0
4,121,Baane Mini Imports,Bergulfsen,Jonas,07-98 9555,Erling Skakkes gate 78,None,Stavern,None,4110,Norway,1504.0,81700.0


### Step 4: Transform Data

In [9]:
# Remove duplicates
c.execute('''DELETE FROM customers
            WHERE customerNumber NOT IN (
                SELECT MIN(customerNumber)
                FROM customers
                GROUP BY customerName, phone, city)
        ''')

In [10]:
# Update missing values
c.execute('''UPDATE customers SET state = "N/A" 
            WHERE state IS NULL''')

In [11]:
# Aggregate data
c.execute('''SELECT city, AVG(creditLimit) AS avg_credit_limit
             FROM customers
             GROUP BY city''')

# Fetch the transformed data
transformed_data = c.fetchall()
transformed_data[0:10]

[('Aachen', 0.0),
 ('Allentown', 100600.0),
 ('Amsterdam', 0.0),
 ('Auckland', 77700.0),
 ('Auckland  ', 99000.0),
 ('Barcelona', 60300.0),
 ('Bergamo', 119600.0),
 ('Bergen', 96800.0),
 ('Berlin', 0.0),
 ('Bern', 0.0)]

In [12]:
# Sanity Check with Pandas
pd.read_sql('''SELECT city, AVG(creditLimit) AS avg_credit_limit
             FROM customers
             GROUP BY city''', conn)

,city,avg_credit_limit
0,Aachen,0.0
1,Allentown,100600.0
2,Amsterdam,0.0
3,Auckland,77700.0
4,Auckland,99000.0
...,...,...
91,Versailles,77900.0
92,Warszawa,0.0
93,Wellington,86800.0
94,White Plains,102700.0


### Step 5: Serve Data

In [13]:
# Save and then close
conn.commit()
conn.close()